In [4]:
import numpy as np
import qiskit_nature
from qiskit_nature.second_q.hamiltonians.lattices import (
    BoundaryCondition,
    LineLattice,
)
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import ParityMapper

import sys
sys.path.append('../../../../')

# qiskit's parity mapper "assumes" (orb1,spin1), ... (orbn,spin1), (orb1,spin2), ..., (orbn,spin2)
# ordering of second quantized orbitals, which is non consistent qiskit's own FermiHubbardModel routine,
# which has (orb1,spin1), (orb1,spin2), ... (orbn,spin1), (orbn,spin2)
# We modifed FermiHubbardModel, which uses ordering of 
# (orb1,spin1), ... (orbn,spin1), (orb1,spin2), ..., (orbn,spin2)
# This is efficient for two-site case only. In other cases, using
# Jordan-Wigner with qiskit's implementation of FermiHubbardModel is better.
from fermi_hubbard_model import FermiHubbardModel

qiskit_nature.settings.use_pauli_sum_op = False

n_site  = 2
n_qubit = 2*n_site-2 # use two qubit reduction
dim = 2**n_qubit
nU  = 5

U_max = 5.0
Us = np.linspace(0.0,5.0,nU)
nU = len(Us)
print(Us)
hamiltonians = []

n_electrons = [1,1]
for U_coulomb in Us:
    t = 1.0
    mu = U_coulomb/2
    bc = BoundaryCondition.PERIODIC
    chain = LineLattice(num_nodes=n_site, boundary_condition=bc, edge_parameter=-t, onsite_parameter=-mu)
    HubbardChain = FermiHubbardModel(chain,onsite_interaction=U_coulomb)
    H = HubbardChain.second_q_op().simplify()
    mapper = ParityMapper(num_particles=n_electrons)
    hamiltonians.append(mapper.map(H))

Us_fine = np.linspace(0.0,5.0,101)
nU_fine = len(Us_fine)
eigen_energies_fine = np.zeros((nU_fine,dim),dtype=float)

iU_fine = 0
for U_coulomb in Us_fine:
    t = 1.0
    mu = U_coulomb/2
    bc = BoundaryCondition.PERIODIC
    chain = LineLattice(num_nodes=n_site, boundary_condition=bc, edge_parameter=-t, onsite_parameter=-mu)
    HubbardChain = FermiHubbardModel(chain,onsite_interaction=U_coulomb)
    h = HubbardChain.second_q_op().simplify()
    mapper = ParityMapper(num_particles=n_electrons)
    h_qubit = mapper.map(h)
    eigen_e, eigen_v = np.linalg.eigh(h_qubit.to_matrix())
    eigen_energies_fine[iU_fine,:] = eigen_e[:]
    iU_fine += 1

n_hamiltonians = nU

for alpha in range(n_hamiltonians):
    print(alpha, hamiltonians[alpha])

hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    print(hamiltonian_diffs[alpha])

hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    print(hamiltonian_diffs_list[alpha])


# exact eigenvalues
eigen_energies_exact  = np.zeros((n_hamiltonians,dim),dtype=float)
eigen_vectors_exact   = np.zeros((n_hamiltonians,dim,dim),dtype=complex)
for alpha in range(n_hamiltonians):
    eigen_e, eigen_v = np.linalg.eigh(hamiltonians[alpha].to_matrix())
    indx = np.argsort(eigen_e.real)
    for i in range(dim):
        eigen_energies_exact[alpha,i]   = eigen_e[indx[i]].real
        eigen_vectors_exact[alpha,:,i] = eigen_v[:,indx[i]]


# degenerate perturbation theory
import copy
nsec = 0
sec_list = []
done = [False]*dim
for i in range(dim):
    if done[i]:
        continue
    else:
        l = [i]
        done[i] =True
        for j in range(i+1,dim):
            if done[j]:
                continue
            else:
                if (np.abs(eigen_energies_exact[0,i]-eigen_energies_exact[0,j])<1E-8):
                    l.append(j)
                    done[j] = True
        sec_list.append(l)
#print(sec_list)
nsec = len(sec_list)
nesec = np.zeros(nsec,dtype=int)
for isec in range(nsec):
    nesec[isec] = len(sec_list[isec])
    #print(nesec[isec])
# diagonalize hamiltonian_diffs[0] for each degenerate sector
hp_mat = (eigen_vectors_exact[0,:,:].conj().transpose())@(hamiltonian_diffs[0].to_matrix())@eigen_vectors_exact[0,:,:]
#deps_sec = []
#w_sec   = []
for isec in range(nsec):
    hp_proj = np.zeros((nesec[isec],nesec[isec]),dtype=complex)
    for i in range(nesec[isec]):
        ii = sec_list[isec][i]
        for j in range(nesec[isec]):
            jj = sec_list[isec][j]
            hp_proj[i,j] = hp_mat[ii,jj]
    deps, w = np.linalg.eigh(hp_proj)
    v_t = np.zeros((dim,nesec[isec]),dtype=complex)

    for i in range(nesec[isec]):
        ii = sec_list[isec][i]
        v_t[:,i] = copy.deepcopy(eigen_vectors_exact[0,:,ii])
    for i in range(nesec[isec]):
        ii = sec_list[isec][i]
        eigen_vectors_exact[0,:,ii] = v_t@w[:,i]
    #print(deps,w)
    #deps_sec.append(deps)
    #w_sec.append(w)
#print(deps_sec,w_sec)
#print(eigen_vectors_exact[0,:,:])


def ExactTimeEvolution (alpha, time):
    Vl = copy.deepcopy(eigen_vectors_exact[alpha,:,:])
    evol = np.zeros((dim,dim),dtype=complex)
    exp_d = np.diag(np.exp(-1j*eigen_energies_exact[alpha,:]*time))
    evol = Vl@exp_d@Vl.conj().T
    return evol

# exact results
norms_exact  = np.ones((n_hamiltonians,dim),dtype=float)
for i in range(dim):
    phi = eigen_vectors_exact[0,:,i]
    for alpha in range(1,n_hamiltonians):
        proj_matrix = np.outer(eigen_vectors_exact[alpha,:,i],eigen_vectors_exact[alpha,:,i].conj())
        phi = proj_matrix@phi
        norms_exact[alpha,i] = phi.conj()@phi
        #print(phi.conj()@hamiltonians[alpha].to_matrix()@phi/norms_exact[alpha,i])
    #print('##')


for alpha in range(n_hamiltonians):
    print(norms_exact[alpha], eigen_energies_exact[alpha,:])


print(eigen_vectors_exact[0,:,0])
print(eigen_vectors_exact[0,:,1])
print(eigen_vectors_exact[0,:,2])
print(eigen_vectors_exact[0,:,3])


import random as rd
import pickle


nmc = int(100)
beta = 0.5

# run qzmc;

norms_qzmc               = np.ones((n_hamiltonians,dim),dtype=float)
eigen_energies_qzmc      = np.zeros((n_hamiltonians,dim),dtype=float)
eigen_energies_qzmc[0,:] = eigen_energies_exact[0,:]

eigen_energies_direct      = np.zeros((n_hamiltonians,dim),dtype=float)
eigen_energies_direct[0,:] = eigen_energies_exact[0,:]


result_values_save = [[[] for _ in range(n_hamiltonians)] for _ in range(dim)]


with open('dimer.result.values','r') as file_:
    lines = file_.readlines()
    ind_line = 0
    for i in range(dim):
        for alpha in range(1,n_hamiltonians):
            result_values = []
            line = lines[ind_line]
            ls = line.split()
            for v in ls:
                result_values.append(float(v))
            #print(result_values)
            result_values_save[i][alpha] = result_values
            ind_line += 1


[0.   1.25 2.5  3.75 5.  ]
0 SparsePauliOp(['IX', 'XI'],
              coeffs=[-1.+0.j, -1.+0.j])
1 SparsePauliOp(['IX', 'II', 'XI', 'ZZ'],
              coeffs=[-1.   +0.j, -0.625+0.j, -1.   +0.j, -0.625+0.j])
2 SparsePauliOp(['IX', 'II', 'XI', 'ZZ'],
              coeffs=[-1.  +0.j, -1.25+0.j, -1.  +0.j, -1.25+0.j])
3 SparsePauliOp(['IX', 'II', 'XI', 'ZZ'],
              coeffs=[-1.   +0.j, -1.875+0.j, -1.   +0.j, -1.875+0.j])
4 SparsePauliOp(['IX', 'II', 'XI', 'ZZ'],
              coeffs=[-1. +0.j, -2.5+0.j, -1. +0.j, -2.5+0.j])
SparsePauliOp(['II', 'ZZ'],
              coeffs=[-0.625+0.j, -0.625+0.j])
SparsePauliOp(['II', 'ZZ'],
              coeffs=[-0.625+0.j, -0.625+0.j])
SparsePauliOp(['II', 'ZZ'],
              coeffs=[-0.625+0.j, -0.625+0.j])
SparsePauliOp(['II', 'ZZ'],
              coeffs=[-0.625+0.j, -0.625+0.j])
[('II', (-0.6249999999999998+0j)), ('ZZ', (-0.6249999999999998+0j))]
[('II', (-0.6249999999999998+0j)), ('ZZ', (-0.6249999999999998+0j))]
[('II', (-0.624999999999

/tmp/ipykernel_71429/2254847414.py:157: ComplexWarning: Casting complex values to real discards the imaginary part
  norms_exact[alpha,i] = phi.conj()@phi


In [5]:
hamiltonian_list = []
for alpha in range(n_hamiltonians):
    hamiltonian_list.append(hamiltonians[alpha].to_list())
    print(hamiltonian_list[alpha])

[('IX', (-0.9999999999999997+0j)), ('XI', (-0.9999999999999997+0j))]
[('IX', (-0.9999999999999997+0j)), ('II', (-0.6249999999999998+0j)), ('XI', (-0.9999999999999997+0j)), ('ZZ', (-0.6249999999999998+0j))]
[('IX', (-0.9999999999999997+0j)), ('II', (-1.2499999999999996+0j)), ('XI', (-0.9999999999999997+0j)), ('ZZ', (-1.2499999999999996+0j))]
[('IX', (-0.9999999999999997+0j)), ('II', (-1.8749999999999993+0j)), ('XI', (-0.9999999999999997+0j)), ('ZZ', (-1.8749999999999993+0j))]
[('IX', (-0.9999999999999997+0j)), ('II', (-2.499999999999999+0j)), ('XI', (-0.9999999999999997+0j)), ('ZZ', (-2.499999999999999+0j))]


In [6]:
for i in range(1):
    for alpha in range(1,n_hamiltonians):
        # compute values
        norm    = np.zeros((2),dtype=float)
        dE1     = np.zeros((2),dtype=float)
        dE2     = np.zeros((2),dtype=float)
        i_meas = 0
        n_meas = len(result_values_save[i][alpha])//2
        # 0; norm
        for imc in range(nmc):
            # indx == 0
            norm[0] += result_values_save[i][alpha][i_meas]
            # indx == 1
            norm[1] += result_values_save[i][alpha][i_meas+n_meas]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                # indx == 0
                dE1[0] +=  coeff * result_values_save[i][alpha][i_meas]
                # indx == 1
                dE1[1] +=  coeff * result_values_save[i][alpha][i_meas+n_meas]
                i_meas += 1

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                # indx == 0
                dE2[0] += coeff * result_values_save[i][alpha][i_meas]
                # indx == 1
                dE2[1] += coeff * result_values_save[i][alpha][i_meas+n_meas]
                i_meas += 1
        
        dE1[0]  /=norm[0]
        dE1[1]  /=norm[1]

        dE2[0]  /=norm[0]
        dE2[1]  /=norm[1]

        norm[0]  /=nmc
        norm[1]  /=nmc

        norm_avg = np.sum(norm)/2
        dE1_avg  = np.sum(dE1)/2
        dE2_avg  = np.sum(dE2)/2

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1_avg += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2_avg += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_qzmc[alpha,i] = eigen_energies_qzmc[alpha-1,i] + dE1_avg
        norms_qzmc[alpha,i] = norm_avg

            
        print(alpha, norms_qzmc[alpha,i], eigen_energies_qzmc[alpha,i]-eigen_energies_exact[alpha,i])
        st = '# {i}/{dim}: {percent}%'.format(i=i+1,dim=dim,percent=((alpha)/(n_hamiltonians-1)*100))
        print(st)

1 0.7550491859451142 0.043984079669487475
# 1/4: 25.0%
2 0.5710252324239535 0.12951915389251312
# 1/4: 50.0%
3 0.39236254181369595 0.25544850073110137
# 1/4: 75.0%
4 0.2298170427614632 0.40856055793126966
# 1/4: 100.0%


/tmp/ipykernel_71429/1478120994.py:24: ComplexWarning: Casting complex values to real discards the imaginary part
  dE1[0] +=  coeff * result_values_save[i][alpha][i_meas]
/tmp/ipykernel_71429/1478120994.py:26: ComplexWarning: Casting complex values to real discards the imaginary part
  dE1[1] +=  coeff * result_values_save[i][alpha][i_meas+n_meas]
/tmp/ipykernel_71429/1478120994.py:40: ComplexWarning: Casting complex values to real discards the imaginary part
  dE2[0] += coeff * result_values_save[i][alpha][i_meas]
/tmp/ipykernel_71429/1478120994.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  dE2[1] += coeff * result_values_save[i][alpha][i_meas+n_meas]
/tmp/ipykernel_71429/1478120994.py:67: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_qzmc[alpha,i] = eigen_energies_qzmc[alpha-1,i] + dE1_avg


In [7]:
with open('dimer.qzmc.energies','w') as file_:
    s = '# U, E_k(U), k = 1,..'
    for alpha in range(n_hamiltonians):
        s = '{:.16e}'.format(Us[alpha])
        for i in range(dim):
            s += '  {:.16e}'.format(eigen_energies_qzmc[alpha,i])
        s += '\n'
        file_.write(s)

In [8]:
with open('dimer.qzmc.norms','w') as file_:
    s = '# U, psi^2_k(U), k = 1,..'
    for alpha in range(n_hamiltonians):
        s = '{:.16e}'.format(Us[alpha])
        for i in range(dim):
            s += '  {:.16e}'.format(norms_qzmc[alpha,i])
        s += '\n'
        file_.write(s)

In [9]:
with open('dimer.fine.energies','w') as file_:
    s = '# U, E_{exact,k}(U)'
    for alpha in range(nU_fine):
        s = '{:.16e}'.format(Us_fine[alpha])
        for i in range(dim):
            s += '  {:.16e}'.format(eigen_energies_fine[alpha,i])
        s += '\n'
        file_.write(s)